In [11]:
import torch
import time
torch.__version__

'2.10.0+cu128'

# **CPU vs GPU Cuda:0**

In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU {gpu_name} is available")
else:
    print("GPU is not available")

GPU Tesla T4 is available


In [29]:
dims = 800   # shape of input matrix
low = 5     # lower bound of matrix element
high = 100   # upper bound of matrix element

In [25]:
def calculate_time_cpu(dims,low,high,device=None):
    X = torch.randint(low=low,high=high,size=(dims,dims),device=device,dtype=torch.float32)
    Y = torch.randint(low=low,high=high,size=(dims,dims),device=device,dtype=torch.float32)
    start_time = time.time()
    mul = torch.matmul(X,Y)
    mul_time = time.time() - start_time
    return mul_time

In [26]:
def calculate_time_gpu(dims,low,high,device=None):
    X = torch.randint(low=low,high=high,size=(dims,dims),device=device,dtype=torch.float32)
    Y = torch.randint(low=low,high=high,size=(dims,dims),device=device,dtype=torch.float32)
    start_time = time.time()
    mul = torch.matmul(X,Y)
    torch.cuda.synchronize()
    mul_time = time.time() - start_time
    return mul_time

In [32]:
cpu_time = calculate_time_cpu(dims,low,high,torch.device('cpu'))
gpu_time = calculate_time_gpu(dims,low,high,device)
print(f"CPU Time: {cpu_time}")
print(f"GPU Time: {gpu_time}")
print(f"GPU{device} is {cpu_time/gpu_time} times faster than CPU")

CPU Time: 0.01618814468383789
GPU Time: 0.0005974769592285156
GPUcuda:0 is 27.09417398244214 times faster than CPU


# **Autograd: Automatic Gradient Computation**
Given x, w, b, y
- z = w*x + b
- y_pred = sigmoid(y)
- loss = -[y*ln(y_pred) + (1-y)*ln(1-y_pred)]

In [33]:
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)
w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [38]:
def manual_gradient(x,y,w,b):
    z = w*x + b
    y_pred = torch.sigmoid(z)
    epsilon = 1e-8  # To prevent log(0)
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
    loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
    dl_dypred = (y_pred - y) // (y_pred*(1-y_pred))
    dypred_dz = y_pred*(1-y_pred)
    dz_dw = x
    dz_db = 1
    dl_dw = dl_dypred * dypred_dz * dz_dw
    dl_db = dl_dypred * dypred_dz * dz_db
    return loss, dl_dw, dl_db

In [39]:
loss, wgred, bgred = manual_gradient(x,y,w,b)
print(f"Loss: {loss}")
print(f"Weight gradient: {wgred}")
print(f"Bias gradient: {bgred}")

Loss: 6.701176166534424
Weight gradient: 6.688785076141357
Bias gradient: 0.9983261823654175


In [40]:
w_ = torch.tensor(1.0,requires_grad=True)
b_ = torch.tensor(0.0,requires_grad=True)
z = w_ * x + b_
y_pred = torch.sigmoid(z)
epsilon = 1e-8  # To prevent log(0)
y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred))
loss.backward()

In [42]:
loss, w_.grad, b_.grad

(tensor(6.7012, grad_fn=<NegBackward0>), tensor(6.6918), tensor(0.9988))